# Cleaning Pipeline 05 — Decision Policy, Audit Trail, and Clean Dataset Copy

This notebook combines the previous outputs into one auditable decision table.

**Conservative default policy:** automatically exclude only corrupt files and redundant same-label exact duplicates. Cross-label duplicates, pHash pairs, DINO pairs, and image-quality anomalies remain manual-review candidates.

The original dataset is never modified.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))
DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

MANIFEST_PATH = OUTPUT_ROOT / "00_manifest" / "train_manifest.csv"
QUALITY_PATH = OUTPUT_ROOT / "01_quality" / "image_quality_candidates.csv"
EXACT_PATH = OUTPUT_ROOT / "02_exact_duplicates" / "exact_duplicate_groups.csv"
PHASH_PATH = OUTPUT_ROOT / "03_phash" / "phash_duplicate_pairs.csv"
DINO_PATH = OUTPUT_ROOT / "04_dino_embeddings" / "dino_duplicate_pairs.csv"

manifest = pd.read_csv(MANIFEST_PATH)
def read_optional(path): return pd.read_csv(path) if path.exists() else pd.DataFrame()
quality, exact, phash, dino = map(read_optional, [QUALITY_PATH, EXACT_PATH, PHASH_PATH, DINO_PATH])

print("Manifest:", len(manifest))
print("Quality candidates:", len(quality))
print("Exact duplicate rows:", len(exact))
print("pHash pairs:", len(phash))
print("DINO pairs:", len(dino))

In [ ]:
candidate_rows = []

for _, row in manifest[~manifest["is_valid"]].iterrows():
    candidate_rows.append({
        "path": row["path"], "relative_path": row["relative_path"],
        "class_name": row["class_name"], "label": int(row["label"]),
        "reason": "corrupt_or_unreadable", "default_action": "exclude",
        "evidence": row.get("error", ""),
    })

if len(quality):
    for _, row in quality.iterrows():
        flags = [c for c in quality.columns if c.startswith("flag_") and bool(row.get(c, False))]
        candidate_rows.append({
            "path": row["path"], "relative_path": row["relative_path"],
            "class_name": row["class_name"], "label": int(row["label"]),
            "reason": "image_quality_review", "default_action": "keep_pending_review",
            "evidence": ",".join(flags),
        })

if len(exact):
    for _, row in exact.iterrows():
        default_action = "exclude" if row["recommended_action"] == "remove_duplicate" else ("keep_pending_review" if row["cross_label"] else "keep")
        candidate_rows.append({
            "path": row["path"], "relative_path": row["relative_path"],
            "class_name": row["class_name"], "label": int(row["label"]),
            "reason": "exact_md5_duplicate", "default_action": default_action,
            "evidence": f"group_id={row['group_id']}; cross_label={row['cross_label']}",
        })

for method_name, pairs, score_col in [
    ("phash_near_duplicate", phash, "phash_distance"),
    ("dino_embedding_similarity", dino, "cosine_similarity"),
]:
    if not len(pairs): continue
    for _, row in pairs.iterrows():
        for side, other in [("a","b"),("b","a")]:
            candidate_rows.append({
                "path": row[f"path_{side}"],
                "relative_path": row.get(f"relative_path_{side}", ""),
                "class_name": row[f"class_{side}"],
                "label": int(row[f"label_{side}"]),
                "reason": method_name,
                "default_action": "keep_pending_review",
                "evidence": f"paired_with={row[f'path_{other}']}; {score_col}={row[score_col]}; cross_label={row['cross_label']}",
            })

candidates = pd.DataFrame(candidate_rows).drop_duplicates()
display(candidates["reason"].value_counts().to_frame("rows"))
display(candidates["default_action"].value_counts().to_frame("rows"))

In [ ]:
priority = {"exclude": 3, "keep_pending_review": 2, "keep": 1}

if len(candidates):
    candidates["action_priority"] = candidates["default_action"].map(priority)
    decisions = (
        candidates.sort_values("action_priority", ascending=False)
        .groupby("path", as_index=False)
        .agg({
            "relative_path":"first", "class_name":"first", "label":"first",
            "default_action":"first",
            "reason":lambda v: " | ".join(sorted(set(v))),
            "evidence":lambda v: " || ".join(sorted(set(map(str,v)))),
        })
    )
else:
    decisions = pd.DataFrame(columns=["path","relative_path","class_name","label","default_action","reason","evidence"])

decisions["final_action"] = decisions["default_action"]
decisions["reviewer"] = ""
decisions["review_notes"] = ""
display(decisions.head(30))

## Manual review workflow

Export `review_decisions.csv`, edit `final_action`, then rerun the clean-copy cell.

Allowed decisions:

- `keep`
- `exclude`
- `keep_pending_review`

Resolve or intentionally keep every pending row before final training.

In [ ]:
stage_dir = OUTPUT_ROOT / "05_decisions"
stage_dir.mkdir(parents=True, exist_ok=True)
candidates.to_csv(stage_dir / "all_cleaning_evidence.csv", index=False)
decisions.to_csv(stage_dir / "review_decisions.csv", index=False)
print("Saved review files to", stage_dir)

In [ ]:
APPLY_CLEAN_COPY = False
CLEAN_ROOT = REPO_ROOT / "BDC2026_clean"
COPY_MODE = "symlink"  # "symlink" or "copy"

if APPLY_CLEAN_COPY:
    reviewed = pd.read_csv(stage_dir / "review_decisions.csv")
    exclude_paths = set(reviewed.loc[reviewed["final_action"] == "exclude", "path"])
    CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

    kept = excluded = 0
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Building clean train"):
        src = Path(row["path"])
        dst = CLEAN_ROOT / row["relative_path"]
        if str(src) in exclude_paths:
            excluded += 1
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() or dst.is_symlink():
            continue
        if COPY_MODE == "symlink":
            dst.symlink_to(src.resolve())
        else:
            shutil.copy2(src, dst)
        kept += 1

    test_src, test_dst = DATA_ROOT / "test", CLEAN_ROOT / "test"
    if test_src.exists() and not test_dst.exists():
        if COPY_MODE == "symlink": test_dst.symlink_to(test_src.resolve(), target_is_directory=True)
        else: shutil.copytree(test_src, test_dst)

    sub_src, sub_dst = DATA_ROOT / "submission.csv", CLEAN_ROOT / "submission.csv"
    if sub_src.exists() and not sub_dst.exists():
        if COPY_MODE == "symlink": sub_dst.symlink_to(sub_src.resolve())
        else: shutil.copy2(sub_src, sub_dst)

    print("Clean root:", CLEAN_ROOT)
    print("Kept:", kept, "Excluded:", excluded)
else:
    print("Dry mode. Review the CSV, then set APPLY_CLEAN_COPY=True.")

In [ ]:
audit = pd.Series({
    "original_train_images": len(manifest),
    "default_exclude": int((decisions["default_action"] == "exclude").sum()),
    "pending_manual_review": int((decisions["default_action"] == "keep_pending_review").sum()),
    "explicit_keep_candidates": int((decisions["default_action"] == "keep").sum()),
})
display(audit.to_frame("count"))
audit.to_csv(stage_dir / "cleaning_audit_summary.csv", header=["count"])